In [ ]:
!pip install -q hmmlearn

In [ ]:
import logging
logging.disable()
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from hmmlearn.hmm import GaussianHMM, MultinomialHMM
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.datasets import fetch_openml, load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

In [ ]:
class HMMClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, hmm, width):
        self.hmm = hmm
        self.width = width

    def fit(self, X, y):
        self.models_ = []
        for c in [0, 1]:
            rows = X[y == c]
            lengths = [X.shape[1] // self.width] * len(rows)
            self.models_.append(clone(self.hmm).fit(rows.reshape(-1, self.width), lengths))
        return self

    def decision_function(self, X):
        zero, one = self.models_
        sequences = [x.reshape(-1, self.width) for x in X]
        return np.array([one.score(s) - zero.score(s) for s in sequences])

    def predict(self, X):
        return (self.decision_function(X) > 0).astype(int)

gaussian = HMMClassifier(GaussianHMM(random_state=42), 1)
multinomial = HMMClassifier(MultinomialHMM(random_state=42), 5)

In [ ]:
def evaluate(name, model, experiment, X_test):
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(name, experiment, "Accuracy:", accuracy)
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    results.append([name, experiment, accuracy])
    if name not in best or accuracy > best[name][0]:
        history = pd.DataFrame([m.monitor_.history for m in model.models_], index=["class 0", "class 1"]).T
        best[name] = accuracy, y_test, y_pred, model.decision_function(X_test), history

In [ ]:
def run(name, model, X_train, X_test):
    model.fit(X_train, y_train)
    evaluate(name, model, f"{test_size} default", X_test)
    grid = GridSearchCV(model, {"hmm__n_components": [2, 3, 4, 5]}, cv=5)
    grid.fit(X_train, y_train)
    print(grid.best_params_)
    evaluate(name, grid.best_estimator_, f"{test_size} tuned", X_test)

In [ ]:
def draw():
    for name in best:
        accuracy, y_true, y_pred, scores, history = best[name]
        ConfusionMatrixDisplay.from_predictions(y_true, y_pred, cmap="Blues")
        plt.title(f"{name} {accuracy:.3f}")
        history.plot(title=f"{name} training curve (log likelihood)", xlabel="EM iteration")
        (-history).plot(title=f"{name} loss curve", xlabel="EM iteration")
        plt.show()
    for name in best:
        accuracy, y_true, y_pred, scores, history = best[name]
        fpr, tpr, thresholds = roc_curve(y_true, scores)
        plt.plot(fpr, tpr, label=f"{name} AUC = {auc(fpr, tpr):.3f}")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title("ROC curve")
    plt.legend()
    plt.show()
    table = pd.DataFrame(results, columns=["classifier", "experiment", "accuracy"])
    return table.pivot_table(index="experiment", columns="classifier", values="accuracy").round(3)

In [ ]:
dataset = fetch_openml("ionosphere", version=1, as_frame=False)
X, y = dataset.data, (dataset.target == "g").astype(int)
X = X[:, X.std(axis=0) > 0]
results, best = [], {}

for test_size in [0.5, 0.4, 0.3, 0.2]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42, stratify=y)
    scaler = StandardScaler()
    run("GaussianHMM", gaussian, scaler.fit_transform(X_train), scaler.transform(X_test))
    bins = KBinsDiscretizer(n_bins=5, encode="onehot-dense", strategy="uniform")
    run("MultinomialHMM", multinomial, bins.fit_transform(X_train).astype(int), bins.transform(X_test).astype(int))
draw()

In [ ]:
dataset = load_breast_cancer()
X, y = dataset.data, dataset.target
results, best = [], {}

for test_size in [0.5, 0.4, 0.3, 0.2]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42, stratify=y)
    scaler = StandardScaler()
    run("GaussianHMM", gaussian, scaler.fit_transform(X_train), scaler.transform(X_test))
    bins = KBinsDiscretizer(n_bins=5, encode="onehot-dense", strategy="uniform")
    run("MultinomialHMM", multinomial, bins.fit_transform(X_train).astype(int), bins.transform(X_test).astype(int))
draw()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torchvision.datasets import MNIST, CIFAR10
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay, roc_curve, auc

device = "cuda" if torch.cuda.is_available() else "cpu"
epochs = 5
print(device)

In [ ]:
def CNN():
    return nn.Sequential(
        nn.LazyConv2d(32, 3), nn.ReLU(), nn.MaxPool2d(2),
        nn.LazyConv2d(64, 3), nn.ReLU(), nn.MaxPool2d(2),
        nn.LazyConv2d(64, 3), nn.ReLU(),
        nn.Flatten(),
        nn.LazyLinear(64), nn.ReLU(),
        nn.LazyLinear(10),
    )

In [ ]:
def conv(channels, size):
    return nn.Sequential(nn.LazyConv2d(channels, size, padding="same"), nn.LazyBatchNorm2d(), nn.ReLU())

def VGG16():
    layers = []
    for count, channels in [(2, 64), (2, 128), (3, 256), (3, 512), (3, 512)]:
        for i in range(count):
            layers.append(conv(channels, 3))
        layers.append(nn.MaxPool2d(2))
    return nn.Sequential(
        *layers,
        nn.Flatten(),
        nn.LazyLinear(512), nn.ReLU(),
        nn.LazyLinear(512), nn.ReLU(),
        nn.LazyLinear(10),
    )

In [ ]:
def AlexNet():
    return nn.Sequential(
        conv(64, 3), nn.MaxPool2d(2),
        conv(192, 5), nn.MaxPool2d(2),
        conv(384, 3), conv(256, 3), conv(256, 3), nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Dropout(), nn.LazyLinear(1024), nn.ReLU(),
        nn.Dropout(), nn.LazyLinear(1024), nn.ReLU(),
        nn.LazyLinear(10),
    )

In [ ]:
class Inception(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.branches = nn.ModuleList([
            conv(n, 1),
            nn.Sequential(conv(n, 1), conv(n, 3)),
            nn.Sequential(conv(n, 1), conv(n, 5)),
            nn.Sequential(nn.MaxPool2d(3, stride=1, padding=1), conv(n, 1)),
        ])

    def forward(self, x):
        return torch.cat([branch(x) for branch in self.branches], dim=1)

def GoogLeNet():
    return nn.Sequential(
        conv(64, 3), nn.MaxPool2d(2),
        Inception(32), Inception(64), nn.MaxPool2d(2),
        Inception(64), Inception(128), nn.MaxPool2d(2),
        Inception(128),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        nn.LazyLinear(10),
    )

In [ ]:
class RNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(X.shape[1] * 32, 128, batch_first=True)
        self.linear = nn.Linear(128, 10)

    def forward(self, x):
        rows = x.transpose(1, 2).flatten(2)
        output, state = self.lstm(rows)
        return self.linear(output[:, -1])

In [ ]:
def predict(model, X):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(batch.to(device)).cpu() for batch in X.split(500)])

def fit(model):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters())
    loss_function = nn.CrossEntropyLoss()
    history = []
    for epoch in range(epochs):
        model.train()
        for rows in torch.randperm(len(X_train)).split(128):
            loss = loss_function(model(X_train[rows].to(device)), y_train[rows].to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        record = []
        for X_part, y_part in [(X_train, y_train), (X_val, y_val)]:
            scores = predict(model, X_part)
            record.append(loss_function(scores, y_part).item())
            record.append(accuracy_score(y_part, scores.argmax(1)))
        history.append(record)
    return pd.DataFrame(history, columns=["train loss", "train accuracy", "val loss", "val accuracy"])

In [ ]:
def evaluate(name, model, experiment):
    history = fit(model)
    scores = predict(model, X_test).softmax(1)
    y_pred = scores.argmax(1)
    accuracy = accuracy_score(y_test, y_pred)
    print(name, experiment, "Accuracy:", accuracy)
    print(classification_report(y_test, y_pred))
    results.append([name, experiment, accuracy])
    if name not in best or accuracy > best[name][0]:
        best[name] = accuracy, y_test, y_pred, scores, history

In [ ]:
def draw():
    for name in best:
        accuracy, y_true, y_pred, scores, history = best[name]
        ConfusionMatrixDisplay.from_predictions(y_true, y_pred, cmap="Blues")
        plt.title(f"{name} {accuracy:.3f}")
        history[["train accuracy", "val accuracy"]].plot(title=f"{name} training curve", xlabel="epoch")
        history[["train loss", "val loss"]].plot(title=f"{name} loss curve", xlabel="epoch")
        plt.show()
    for name in best:
        accuracy, y_true, y_pred, scores, history = best[name]
        y_true = pd.get_dummies(y_true.numpy()).values.ravel()
        fpr, tpr, thresholds = roc_curve(y_true, scores.ravel())
        plt.plot(fpr, tpr, label=f"{name} AUC = {auc(fpr, tpr):.3f}")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title("ROC curve")
    plt.legend()
    plt.show()
    table = pd.DataFrame(results, columns=["classifier", "experiment", "accuracy"])
    return table.pivot_table(index="experiment", columns="classifier", values="accuracy").round(3)

In [ ]:
train = MNIST("data", train=True, download=True)
test = MNIST("data", train=False, download=True)
X = np.concatenate([train.data, test.data])[:20000]
y = np.concatenate([train.targets, test.targets])[:20000]
X = torch.tensor(X).float() / 255
X = X.reshape(20000, 1, 28, 28)
X = nn.functional.pad(X, (2, 2, 2, 2))
y = torch.tensor(y)
print(X.shape)

In [ ]:
results, best = [], {}

for test_size in [0.4, 0.3, 0.2]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train)
    for build in [CNN, VGG16, RNN, AlexNet, GoogLeNet]:
        evaluate(build.__name__, build(), f"test {test_size}")
draw()

In [ ]:
train = CIFAR10("data", train=True, download=True)
test = CIFAR10("data", train=False, download=True)
X = np.concatenate([train.data, test.data])[:20000]
y = np.concatenate([train.targets, test.targets])[:20000]
X = torch.tensor(X).float() / 255
X = X.permute(0, 3, 1, 2)
y = torch.tensor(y)
print(X.shape)

In [ ]:
results, best = [], {}

for test_size in [0.4, 0.3, 0.2]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train)
    for build in [CNN, VGG16, RNN, AlexNet, GoogLeNet]:
        evaluate(build.__name__, build(), f"test {test_size}")
draw()